# Fake News Detection — Notebook 3: BERT Fine-Tuning

Fine-tune `bert-base-uncased` for binary fake-news classification.

**Input:** `data/lemmatized.csv`  
**Output:** `bert-finetuned/` (saved model + tokenizer)

> **Requirements:** `pip install transformers torch`  
> A GPU is strongly recommended (CUDA or Apple MPS). Training on CPU is possible but slow.

---

## 0. Setup

In [1]:
import pandas as pd
import numpy as np
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

device = torch.device('cuda' if torch.cuda.is_available() else
                      'mps'  if torch.backends.mps.is_available() else
                      'cpu')
print(f'Using device: {device}')


Using device: mps


## 1. Load Data

In [2]:
df = pd.read_csv('../data/lemmatized.csv')
df = df[['text', 'label']].dropna().reset_index(drop=True)
df['label'] = df['label'].astype(int)

print(f'Samples: {len(df)}')
print(f'Label distribution:\n{df["label"].value_counts()}')


Samples: 71349
Label distribution:
label
1    36323
0    35026
Name: count, dtype: int64


## 2. Tokenizer & Model

In [3]:
MODEL_NAME = 'bert-base-uncased'

tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2
).to(device)

print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model parameters: 109,483,778


## 3. Dataset Class

In [4]:
class FakeNewsDataset(Dataset):
    """Tokenises on-the-fly to avoid storing the full tensor matrix in memory."""

    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts     = list(texts)
        self.labels    = list(labels)
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt',
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long),
        }


## 4. Train / Test Split & DataLoaders

In [6]:
BATCH_SIZE = 16
MAX_LEN    = 128

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['text'], df['label'], test_size=0.2, random_state=42, stratify=df['label']
)

train_dataset = FakeNewsDataset(
    train_texts.reset_index(drop=True),
    train_labels.reset_index(drop=True),
    tokenizer, MAX_LEN,
)
test_dataset = FakeNewsDataset(
    test_texts.reset_index(drop=True),
    test_labels.reset_index(drop=True),
    tokenizer, MAX_LEN,
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
test_loader   = DataLoader(test_dataset,   batch_size=BATCH_SIZE, num_workers=0)

print(f'Train batches: {len(train_loader)} | Test batches: {len(test_loader)}')


Train batches: 3568 | Test batches: 892


## 5. Fine-Tuning Loop

3 epochs with AdamW (lr = 2e-5) — standard BERT fine-tuning recipe.

In [7]:
EPOCHS = 3
optimizer = AdamW(model.parameters(), lr=2e-5)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch_idx, batch in enumerate(train_loader):
        optimizer.zero_grad()

        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if (batch_idx + 1) % 200 == 0:
            print(f'  Epoch {epoch+1} | Batch {batch_idx+1}/{len(train_loader)} '
                  f'| Running loss: {total_loss/(batch_idx+1):.4f}')

    avg_loss = total_loss / len(train_loader)
    train_dataset = torch.utils.data.Subset(train_dataset, range(500))
    print(f'Epoch {epoch+1}/{EPOCHS} complete | Avg loss: {avg_loss:.4f}\n')


  Epoch 1 | Batch 200/3568 | Running loss: 0.3650
  Epoch 1 | Batch 400/3568 | Running loss: 0.2584
  Epoch 1 | Batch 600/3568 | Running loss: 0.2153
  Epoch 1 | Batch 800/3568 | Running loss: 0.1896
  Epoch 1 | Batch 1000/3568 | Running loss: 0.1753
  Epoch 1 | Batch 1200/3568 | Running loss: 0.1615
  Epoch 1 | Batch 1400/3568 | Running loss: 0.1515
  Epoch 1 | Batch 1600/3568 | Running loss: 0.1436
  Epoch 1 | Batch 1800/3568 | Running loss: 0.1373
  Epoch 1 | Batch 2000/3568 | Running loss: 0.1315
  Epoch 1 | Batch 2200/3568 | Running loss: 0.1268
  Epoch 1 | Batch 2400/3568 | Running loss: 0.1234
  Epoch 1 | Batch 2600/3568 | Running loss: 0.1195
  Epoch 1 | Batch 2800/3568 | Running loss: 0.1157
  Epoch 1 | Batch 3000/3568 | Running loss: 0.1118
  Epoch 1 | Batch 3200/3568 | Running loss: 0.1093
  Epoch 1 | Batch 3400/3568 | Running loss: 0.1063
Epoch 1/3 complete | Avg loss: 0.1044

  Epoch 2 | Batch 200/3568 | Running loss: 0.0364
  Epoch 2 | Batch 400/3568 | Running loss: 0.037

## 6. Evaluation

In [8]:
model.eval()
preds, true_vals = [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
        true_vals.extend(batch['labels'].numpy())

print(f'Accuracy: {accuracy_score(true_vals, preds):.4f}\n')
print(classification_report(true_vals, preds, target_names=['Real (0)', 'Fake (1)']))


Accuracy: 0.9819

              precision    recall  f1-score   support

    Real (0)       0.98      0.98      0.98      7005
    Fake (1)       0.98      0.98      0.98      7265

    accuracy                           0.98     14270
   macro avg       0.98      0.98      0.98     14270
weighted avg       0.98      0.98      0.98     14270



## 7. Save Model

In [9]:
SAVE_PATH = '../bert-finetuned'
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f'Model and tokenizer saved to {SAVE_PATH}/')


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer saved to ../bert-finetuned/


## 8. Inference Helper

Load the saved model and classify a single piece of text.

In [10]:
from transformers import pipeline

classifier = pipeline(
    'text-classification',
    model=SAVE_PATH,
    tokenizer=SAVE_PATH,
    device=0 if device.type == 'cuda' else -1,
)

samples = [
    'Scientists confirm new vaccine shows 95% efficacy in large-scale trial.',
    'BREAKING: government secretly puts microchips in drinking water!!',
]
for text in samples:
    result = classifier(text, truncation=True, max_length=128)[0]
    label  = 'FAKE' if result['label'] == 'LABEL_1' else 'REAL'
    print(f'{label} ({result["score"]:.3f})  |  {text[:80]}')


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

REAL (0.819)  |  Scientists confirm new vaccine shows 95% efficacy in large-scale trial.
FAKE (1.000)  |  BREAKING: government secretly puts microchips in drinking water!!
